# ATIS classifier — train on Colab GPU with more Kaggle data

End-to-end: pull tyre datasets from Kaggle, rebuild the `normal`/`cracked` split
with `prepare_dataset.py` (dedup + leakage-checked 70/15/15), train the YOLOv11
classifier, evaluate + sweep the safety threshold, then download `best.pt`.

**Runtime:** set **Runtime → Change runtime type → GPU (T4)** before running.

After downloading, on your Mac drop `best.pt` + `model_card.json` at the LFS path
`runs/classify/runs/classify/ATIS_Project/tyre_safety_model/weights/` and commit
via Git LFS (see the last cell).

## 1. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU (T4)'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Install dependencies

In [ ]:
%pip install -q ultralytics imagehash kaggle

## 3. Kaggle auth
Upload your `kaggle.json` (Kaggle → Account → Create New API Token).

In [ ]:
import os
from google.colab import files
print('Select your kaggle.json ...')
up = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(up['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets list -s tyre --max-size 1 >/dev/null 2>&1 && echo 'Kaggle auth OK' || echo 'Kaggle auth FAILED'

## 4. Get the ATIS data-prep + training scripts
Clone the public repo **without** LFS (we don't need the 200 MB weights here).

In [ ]:
import os
os.environ['GIT_LFS_SKIP_SMUDGE'] = '1'
!git clone --depth 1 https://github.com/70137131-jpg/ATIS.git /content/ATIS
%cd /content/ATIS

## 5. Build the dataset from Kaggle sources
`--kaggle` is a **global** flag: every `--source` must be a Kaggle slug in one run.
`prepare_dataset.py` pools them, maps folder names to `normal`/`cracked`, drops
corrupt + near-duplicate images, and writes a fresh leakage-checked 70/15/15 split.

Edit the slug list to add/remove sources. The three below = the original set plus a
real-workshop set (the kind of varied data that reduces good-tyre false-flagging).

In [ ]:
!python3 prepare_dataset.py \
  --source jehanbhathena/tire-texture-image-recognition \
  --source warcoder/tyre-quality-classification \
  --source sameersambhare1/tyre-condition-classification-dataset \
  --kaggle

## 6. Train

In [ ]:
# train_model.py: yolo11n-cls, 100 epochs (early-stop patience 20), imgsz 224.
# Writes best.pt + model_card.json under runs/classify/ATIS_Project/tyre_safety_model/
!YOLO_DEVICE=0 python3 train_model.py

## 7. Evaluate + sweep the safety threshold
Held-out test accuracy, cracked recall, good-tyre false-flag rate, and the
smallest `min P(normal)` threshold that still hits the target cracked-recall.

In [ ]:
!python3 evaluate_model.py

## 8. Download the trained weights + model card

In [ ]:
import glob, shutil
from google.colab import files
run = 'runs/classify/ATIS_Project/tyre_safety_model'
best = f'{run}/weights/best.pt'
assert glob.glob(best), f'best.pt not found under {run}/weights/'
shutil.make_archive('/content/atis_model', 'zip', run)
print('Zipped:', run, '-> atis_model.zip (contains weights/best.pt + model_card.json)')
files.download('/content/atis_model.zip')

## 9. Back on your Mac — install the new weights (Git LFS)
```bash
cd ~/Desktop/ATIS
# unzip the download into the LFS-tracked (doubly-nested) path the app resolves
DEST=runs/classify/runs/classify/ATIS_Project/tyre_safety_model
mkdir -p "$DEST/weights"
unzip -o ~/Downloads/atis_model.zip -d "$DEST"

git lfs status                 # best.pt should show as LFS
git add "$DEST/weights/best.pt" "$DEST/model_card.json"
python3 -m pytest tests/test_model_contract.py -q   # sanity: new weights load
git commit -m "feat(model): retrain classifier on expanded tyre dataset"
git push origin main
```